# Practical 6 — Transfer Learning Using VGG16

Use a pretrained VGG16 convolutional base as a feature extractor and train a new classifier for CIFAR-10.

## Aim
Demonstrate transfer learning with a pretrained VGG16 CNN on CIFAR-10.

## Important academic clarification
VGG16 is a classification CNN/feature extractor, not a complete object detector by itself. This notebook follows the common transfer-learning workflow of loading pretrained ImageNet weights, freezing the convolutional base, adding a classifier, and training on CIFAR-10. It performs **image classification**, not bounding-box localization. A genuine object-detection task requires a detection architecture/head such as Faster R-CNN or SSD.

In [ ]:
# 1. Import libraries
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Resizing, Lambda, GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.optimizers import Adam

In [ ]:
# 2. Load CIFAR-10 and define class names
class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# Small subsets reduce runtime for a laptop demonstration
x_train, y_train = x_train[:5000], y_train[:5000]
x_test, y_test = x_test[:1000], y_test[:1000]
y_train = y_train.flatten()
y_test = y_test.flatten()

print("Training images:", x_train.shape)
print("Testing images:", x_test.shape)

In [ ]:
# 3. Load VGG16 pretrained on ImageNet
base_model = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(96, 96, 3)
)

# Freeze pretrained feature-extraction layers
base_model.trainable = False

In [ ]:
# 4. Build transfer-learning model
model = Sequential([
    Input(shape=(32, 32, 3)),
    Resizing(96, 96),
    Lambda(preprocess_input),
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation="relu"),
    Dropout(0.3),
    Dense(10, activation="softmax")
])

# 5. Compile model
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

In [ ]:
# 6. Train the custom classification head
history = model.fit(
    x_train, y_train,
    epochs=3,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

In [ ]:
# 7. Evaluate
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)
print("Test loss:", round(test_loss, 4))
print("Test accuracy:", round(test_accuracy * 100, 2), "%")

In [ ]:
# 8. Plot training and validation curves
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Training accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.title("Transfer Learning Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.title("Transfer Learning Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# 9. Predict and display one test image
sample_index = 0
image = x_test[sample_index]
actual_label = int(y_test[sample_index])

probabilities = model.predict(np.expand_dims(image, axis=0), verbose=0)[0]
predicted_label = int(np.argmax(probabilities))
confidence = float(probabilities[predicted_label] * 100)

plt.figure(figsize=(4, 4))
plt.imshow(image)
plt.title(
    f"Actual: {class_names[actual_label]}\n"
    f"Predicted: {class_names[predicted_label]} ({confidence:.1f}%)"
)
plt.axis("off")
plt.show()

print("Actual class:", class_names[actual_label])
print("Predicted class:", class_names[predicted_label])
print("Predicted-class score:", round(confidence, 2), "%")

## Result
VGG16 pretrained on ImageNet is used as a frozen feature extractor. A custom classifier is trained on a subset of CIFAR-10, and the model is evaluated and used to predict an image class.